# ExioML × pymrio end-to-end demo

Run the PyPI build of ExioML (instead of local sources) to ingest MRIO-style factors, preprocess mixed categorical/numeric features, and train a small model end-to-end.

In [1]:
# Install PyPI builds to mirror user-facing behavior
%pip install -q exioml==0.2.1 pymrio scikit-learn pandas

import exioml
import pandas as pd
from exioml.datasets import prepare_dataset

print('exioml version:', exioml.__version__)

Note: you may need to restart the kernel to use updated packages.
exioml version: 0.2.1


## Quick start: categorical + numeric preprocessing

A tiny synthetic frame showing leave-one-out target encoding for categorical features, plus scaling and train/validation/test splits.

In [2]:
frame = pd.DataFrame({
    'region': ['US','US','CN','CN','CN'],
    'sector': ['A','B','A','B','C'],
    'value': [1.0,2.0,3.0,4.0,5.0],
    'target': [10,20,30,40,50],
})

# Prepare dataset with categorical leave-one-out encoding
splits, preproc = prepare_dataset(
    frame,
    feature_cols=['region','sector','value'],
    target_col='target',
    categorical_cols=['region','sector'],
    ratios=(0.6,0.2,0.2),
    stratify=False,
)

print('train/val/test shapes:', splits.x_train.shape, splits.x_val.shape, splits.x_test.shape)

train/val/test shapes: (3, 3) (1, 3) (1, 3)


## Full pipeline: pymrio sample → tidy factors → ML training

Steps:
1. Load the built-in `pymrio.load_test()` MRIO system and compute derived accounts.
2. Tidy the emission extension into a tabular frame with categorical + numeric helpers.
3. Run ExioML preprocessing (leave-one-out encoding + scaling) and split the data.
4. Train a small gradient boosting regressor and inspect metrics/predictions.

In [3]:
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import pymrio

mrio = pymrio.load_test()
mrio.calc_all()

ext = mrio.emissions.F

factors = (
    ext.stack(['region','sector'])
    .reset_index()
    .rename(columns={0: 'emission'})
)
# Simple numeric helpers derived from categorical columns
factors['region_code'] = pd.factorize(factors['region'])[0].astype('float32')
factors['sector_code'] = pd.factorize(factors['sector'])[0].astype('float32')

sample_size = min(120, len(factors))
sample = factors.sample(sample_size, random_state=0).reset_index(drop=True)
feature_cols = ['region', 'sector', 'region_code', 'sector_code']

splits_mrio, preproc_mrio = prepare_dataset(
    sample,
    feature_cols=feature_cols,
    target_col='emission',
    categorical_cols=['region','sector'],
    ratios=(0.7, 0.15, 0.15),
    stratify=False,
)

model = HistGradientBoostingRegressor(random_state=0)
model.fit(splits_mrio.x_train, splits_mrio.y_train)

def evaluate(name, X, y):
    preds = model.predict(X)
    return {
        'split': name,
        'mae': mean_absolute_error(y, preds),
        'r2': r2_score(y, preds),
    }

metrics = [
    evaluate('train', splits_mrio.x_train, splits_mrio.y_train),
    evaluate('val', splits_mrio.x_val, splits_mrio.y_val),
    evaluate('test', splits_mrio.x_test, splits_mrio.y_test),
]
print(pd.DataFrame(metrics))

example_rows = sample.head(3).copy()
X_new = preproc_mrio.transform(example_rows[feature_cols])
preds = model.predict(X_new)
preview = example_rows.assign(predicted_emission=preds)

print()
print("Sample predictions using the fitted preprocessing + model:")
display(preview)


   split           mae        r2
0  train  1.093809e+07  0.466822
1    val  6.871382e+06  0.490132
2   test  1.599221e+07  0.205101

Sample predictions using the fitted preprocessing + model:


/var/folders/fd/nsvs1bc97kvglzhh5bvwqwqm0000gn/T/ipykernel_42632/2474344813.py:13: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ext.stack(['region','sector'])


,stressor,compartment,region,sector,emission,region_code,sector_code,predicted_emission
0,emission_type1,air,reg4,manufactoring,39138405.0,3.0,2.0,2.444780e+07
1,emission_type2,water,reg5,construction,1071128.2,4.0,4.0,-5.687425e+06
2,emission_type1,air,reg1,manufactoring,23613787.0,0.0,2.0,9.755863e+06
